# Recommendation System for web articles in Mondadori Group
---

In [4]:
# importing
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

In [5]:
# setting
model_name = "nickprock/sentence-bert-base-italian-xxl-uncased"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(32102, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [7]:
def mean_pooling(model_output, attention_mask):
    """
    Apply mean pooling on token embeddings, considering the attention mask
    """
    token_embeddings = model_output[0] # all token embeddings
    # expand attenion mask to multiply the embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    # weighted sum and normalize to obtain averaged sentence embedding
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask

# Definisci alcune frasi di esempio
texts = [
    "Una ragazza si acconcia i capelli.",
    "Una ragazza si sta spazzolando i capelli.",
    "La prossima settimana accorcerò un po' i capelli: sono diventati troppo lunghi.",
    "Mi piacciono i libri gialli.",
    "Sono appassionato di UFO.",
    "La signora in giallo è la mia serie preferita."
]

# Tokenizza le frasi con padding e troncamento
encoded_input = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
# Sposta tutti i tensori sul device scelto
encoded_input = {key: tensor.to(device) for key, tensor in encoded_input.items()}

# Esegui il modello in modalità inference
with torch.no_grad():
    model_output = model(**encoded_input)

# Applica il mean pooling per ottenere un embedding per ciascuna frase
sentence_embeddings = mean_pooling(model_output, encoded_input["attention_mask"])

In [8]:
# Frase di input arbitraria
input_sentence = "ET è il mio film preferito."

# Tokenizza e calcola l'embedding per la frase di input
encoded_input_sentence = tokenizer(input_sentence, padding=True, truncation=True, return_tensors="pt")
# Sposta i tensori sullo stesso device usato precedentemente (ad es. "mps" o "cpu")
encoded_input_sentence = {key: tensor.to(device) for key, tensor in encoded_input_sentence.items()}


In [9]:
# Esegui il modello in modalità inference
with torch.no_grad():
    model_output = model(**encoded_input_sentence)

In [10]:
input_embedding = mean_pooling(model_output, encoded_input_sentence["attention_mask"])
cosine_similarities = F.cosine_similarity(input_embedding, sentence_embeddings, dim=1)
euclidean_distances = torch.norm(sentence_embeddings - input_embedding, dim=1)

In [11]:
cosine_similarities, euclidean_distances

(tensor([0.0381, 0.0205, 0.0947, 0.3218, 0.2086, 0.3362], device='mps:0'),
 tensor([22.9633, 22.9107, 21.5607, 18.5450, 20.1156, 18.3153], device='mps:0'))

---
---
---
---
---

## Testing

In [13]:
from pydantic import BaseModel, Field
from typing import List


class Section(BaseModel):
    title: str = Field(description="main topic of this section of the document")
    start_index: int = Field(description="line number where the section begins")
    end_index: int = Field(description="line number where the section ends")


class StructuredDocument(BaseModel):
    """obtains meaningful sections, each centered around a single concept/topic"""

    sections: List[Section] = Field(description="a list of sections of the document")

In [14]:
def doc_with_lines(document):
    document_lines = document.split("\n")
    document_with_line_numbers = ""
    line2text = {}
    for i, line in enumerate(document_lines):
        document_with_line_numbers += f"[{i}] {line}\n"
        line2text[i] = line
    return document_with_line_numbers, line2text

In [15]:
example_of_text = """
Prosegue lo scontro tra governo e sindacati, ma anche tra sindacati e imprese, sulla riforma del mercato del lavoro.
Il nodo della discussione è la possibile modifica dell'articolo 18.
La norma nello Statuto dei lavoratori del 1970 vieta i licenziamenti in mancanza di giusta causa nelle aziende con più di 15 dipendenti.
Una decisione difficile, una sfida che tanti governi precedenti hanno tentato di vincere, senza successo, e nella quale il governo tecnico
guidato da Mario Monti non intende fallire. Il premier ha reso noto che si andrà avanti nonostante tutto, anche se i sindacati non saranno
daccordo, anche se non si dovesse trovare un accordo unanime. Il mercato e l'Europa chiedono all'Italia un
cambiamento netto nel mercato del lavoro e il terreno di scontro più aspro è quello che riguarda l'articolo 18.
LEGGI ANCHE: Gravidanza e licenziamento
L'articolo 18: Qual è la situazione attuale
Varato nel 1970,  l'articolo 18 afferma che, nelle aziende che hanno più di 15 dipendenti, i lavoratori possono essere licenziati solo per una
giusta causa o un valido motivo.
Se il giudice dovesse valutare che il licenziamento (LEGGI) è stato illeggitimo e in violazione dell'articolo 18 può obbligare
l'azienda al reintegro del lavoratore nella stessa posizione che occupava prima del licenziamento o, se il lavoratore è daccordo,
può liquidarlo con un'indennità pari a 15 mensilità dell'ultimo stipendio o un'indennità crescente con
l'anzianità di servizio.Attualmente quasi due terzi dei lavoratori sono tutelati dall'articolo 18.Cosa cambia con la nuova riforma
L'articolo 18 verrà modificato. Il reintegro sarà obbligatorio solo se il licenziamento è stato discriminatorio,
se l'azienda licenzia per motivi economici sarà obbligata a versare un indennizzo variabile tra 15 e 27 mensilità; se il
licenziamento è causato da motivi disciplinari l'ultima parola spetterà al giudice che potrà decidere se optare per il
reintegro o un indennizzo. Il nuovo articolo 18 si applicherà a tutte le aziende, indipendentemente dal numero di lavoratori.
LEGGI ANCHE: Figli, casa, lavoro, non ce la faccio più.
Le novità per le famiglie e i giovaniLa riforma del lavoro proposta dal governo modifica anche le regole dei contratti a tempo determinato,
che non potranno essere reiterati per più di 36 mesi e sui contratti
a tempo determinato graverà un aumento dei contributi dell'1,4% a carico delle imprese che servirà per finanziare un nuovo sistema
di ammortizzatori sociali. Viene introdotto un nuovo ammortizzatore sociale, chiamato Aspi (assicurazione per l'impiego), che
sostituirà l'assegno di disoccupazione: avrà la durata di un anno per i lavoratori fino a 54 anni, per un massimo di 1.119
euro,è e di 18 mesi per i lavoratori oltre i 54 anni.  Per quanto riguarda le donne saranno vietate le dimissioni in bianco,
un'odiosa pratica messa in atto illegalmente da numerose aziende al momento dell'assunzione, e le aziende verranno invitate a sperimentare
i congedi di paternità obbligatori (LEGGI) per tre anni. Fonte: Liquida
"""

In [21]:
import instructor
import cohere

# Apply the patch to the cohere client
# enables response_model keyword
client = instructor.from_cohere(cohere.Client())


system_prompt = f"""\
You are a world class educator working on organizing your lecture notes.
Read the document below and extract a StructuredDocument object from it where each section of the document is centered around a single concept/topic that can be taught in one lesson.
Each line of the document is marked with its line number in square brackets (e.g. [1], [2], [3], etc). Use the line numbers to indicate section start and end.
"""


def get_structured_document(document_with_line_numbers) -> StructuredDocument:
    return client.chat.completions.create(
        model="command-r-plus",
        response_model=StructuredDocument,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": document_with_line_numbers,
            },
        ],
    )  # type: ignore

ModuleNotFoundError: No module named 'instructor'

In [ ]:
def get_sections_text(structured_doc, line2text):
    segments = []
    for s in structured_doc.sections:
        contents = []
        for line_id in range(s.start_index, s.end_index):
            contents.append(line2text.get(line_id, ''))
        segments.append(
            {
                "title": s.title,
                "content": "\n".join(contents),
                "start": s.start_index,
                "end": s.end_index,
            }
        )
    return segments